# Домашняя работа 8. Итеративные методы решения СЛАУ
**Колпакова Дарья Сергеевна, группа J3112, ИСУ 501390**

**Вариант 11**

**Задание:**
Реализовать итеративные методы Якоби и Гаусса-Зейделя с критерием остановки $\|x_k - x_{k-1}\|_2 < 10^{-5}$.
Использовать их для решения уравнения $Ax = b$, где:
- $A = C^T C$ ($C$ — первые 13 столбцов файла `housing.csv`),
- $b$ — вектор с 13-ю компонентами, состоящий из элементов последнего столбца со строк от $(i - 1) \cdot 13$ до $i \cdot 13 - 1$, где $i$ — номер варианта.

Для варианта $i = 11$:
- Строки для вектора $b$: с $130$ по $142$ включительно.

### Теоретическая справка

Представим матрицу $A$ в виде суммы:
$$A = L + D + U$$
где $L$ — строго нижнетреугольная матрица, $D$ — диагональная матрица, $U$ — строго верхнетреугольная матрица.

**Метод Якоби**
Система $Ax = b$ преобразуется к эквивалентному виду:
$$x = -D^{-1}(L + U)x + D^{-1}b$$
Итерационный процесс метода Якоби:
$$x^{k+1} = -D^{-1}(L + U)x^k + D^{-1}b$$
Достаточное условие сходимости: строгое диагональное преобладание матрицы $A$.

**Метод Гаусса-Зейделя**
Система $Ax = b$ преобразуется к виду:
$$x = -(L + D)^{-1}Ux + (L + D)^{-1}b$$
Итерационный процесс метода Гаусса-Зейделя:
$$x^{k+1} = -(L + D)^{-1}Ux^k + (L + D)^{-1}b$$
Достаточное условие сходимости: матрица $A$ симметрична и положительно определена ($A = A^T > 0$). В нашем случае $A = C^T C$, поэтому она является симметричной и положительно полуопределенной (а при линейной независимости столбцов $C$ — положительно определенной), что гарантирует сходимость метода Гаусса-Зейделя.

In [9]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=6, suppress=True)

VARIANT = 11
TOLERANCE = 1e-5
MAX_ITER = 100000
CSV_PATH = "housing.csv"

def read_housing_data(path):
    for header_mode in ["infer", None]:
        df = pd.read_csv(path, header=header_mode)
        numeric_df = df.select_dtypes(include=[np.number])
        if numeric_df.shape[1] == 14:
            return numeric_df.to_numpy(dtype=float)
    raise ValueError("Не удалось получить 14 числовых столбцов.")

# подготовка данных
data = read_housing_data(CSV_PATH)

# матрица C первые 13 столбцов (все строки)
C = data[:, :13]

# матрица A = C^T * C
A = C.T @ C

# вектор b из последнего (14-го) столбца
start_row = (VARIANT - 1) * 13
end_row = VARIANT * 13
b = data[start_row:end_row, 13]

print(f"Размерность матрицы A: {A.shape}")
print(f"Размерность вектора b: {b.shape}")
print(f"Строки для вектора b: с {start_row} по {end_row - 1}")



Размерность матрицы A: (13, 13)
Размерность вектора b: (13,)
Строки для вектора b: с 130 по 142


In [10]:
# реализация методов

def jacobi_method(A, b, tol=1e-5, max_iter=10000):
    n = len(b)
    x = np.zeros(n)

    D = np.diag(A)
    # если на диагонали есть нули, метод Якоби неприменим
    if np.any(D == 0):
        print("Метод Якоби: на диагонали обнаружены нули")
        return None, 0

    L_plus_U = A - np.diag(D)

    for k in range(1, max_iter + 1):
        x_new = (b - L_plus_U @ x) / D

        # критерий остановки
        if np.linalg.norm(x_new - x, 2) < tol:
            return x_new, k

        x = x_new

        # защита от расходимости
        if np.any(np.isnan(x)) or np.any(np.isinf(x)):
            print(f"Метод Якоби разошелся на итерации {k}")
            return None, k

    print("Метод Якоби: достигнуто максимальное число итераций.")
    return x, max_iter


def gauss_seidel_method(A, b, tol=1e-5, max_iter=10000):
    n = len(b)
    x = np.zeros(n)

    # разбиваем A на L, D, U
    L = np.tril(A, -1)
    D = np.diag(np.diag(A))
    U = np.triu(A, 1)

    L_plus_D = L + D

    # проверка обратимости L+D (на диагонали не должно быть нулей)
    if np.any(np.diag(D) == 0):
        print("Метод Гаусса-Зейделя: обнаружены нули на диагонали")
        return None, 0

    for k in range(1, max_iter + 1):
        # решаем систему (L+D) * x_new = b - U * x
        rhs = b - U @ x
        x_new = np.linalg.solve(L_plus_D, rhs)

        if np.linalg.norm(x_new - x, 2) < tol:
            return x_new, k

        x = x_new

        if np.any(np.isnan(x)) or np.any(np.isinf(x)):
            print(f"Метод Гаусса-Зейделя разошелся на итерации {k}")
            return None, k

    print("Метод Гаусса-Зейделя: достигнуто максимальное число итераций.")
    return x, max_iter

In [11]:
# вычисления

print("\nМетод Якоби")
x_jacobi, iters_jacobi = jacobi_method(A, b, tol=TOLERANCE, max_iter=MAX_ITER)
if x_jacobi is not None:
    print(f"Решение найдено за {iters_jacobi} итераций.")
    print("x =", x_jacobi)

print("\nМетод Гаусса-Зейделя")
x_gs, iters_gs = gauss_seidel_method(A, b, tol=TOLERANCE, max_iter=MAX_ITER)
if x_gs is not None:
    print(f"Решение найдено за {iters_gs} итераций.")
    print("x =", x_gs)


Метод Якоби
Метод Якоби разошелся на итерации 336

Метод Гаусса-Зейделя
Решение найдено за 621 итераций.
x = [ 0.003936 -0.000363 -0.033535  0.480714  6.551798 -0.280295 -0.008101
  0.0636    0.002824 -0.001319 -0.004571 -0.000842 -0.025015]


/tmp/ipykernel_1998/2684105183.py:16: RuntimeWarning: overflow encountered in matmul
  x_new = (b - L_plus_U @ x) / D


In [12]:
# проверка
print("\nТочное решение (np.linalg.solve) ---")
try:
    x_exact = np.linalg.solve(A, b)
    print("x =", x_exact)

    if x_gs is not None:
        print(f"Погрешность Гаусса-Зейделя: {np.linalg.norm(x_gs - x_exact, 2):.6e}")
except np.linalg.LinAlgError:
    print("Матрица A вырождена, точное решение найти невозможно.")


Точное решение (np.linalg.solve) ---
x = [ 0.003934 -0.000365 -0.033536  0.480689  6.551661 -0.280245 -0.0081
  0.063634  0.002826 -0.001319 -0.004597 -0.000842 -0.025014]
Погрешность Гаусса-Зейделя: 1.535622e-04
